<a href="https://colab.research.google.com/github/elifkocs/Basic-Concepts/blob/main/basic_concepts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Data preparation

In [13]:
def preprocess_data(filepath): #veri ön işleme fonksiyonu

    data = pd.read_csv(filepath)

    scaler = StandardScaler()

    X = scaler.fit_transform(data.drop('Outcome', axis=1)) #veri setindeki bağımlı değişken düşürülür, bağımsız değişkenler alınır

    y = data['Outcome'].values #bağımlı değişkeni dataframe formunda değil de numpy formunda al demiş olduk

    return X, y

X, y = preprocess_data('/content/diabetes.csv') #veri seti X ve y olarak ayrılmış oldu


X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)#veri setini train ve validasyon seti olarak ikiye ayırmış olduk

# Activation Functions, Weights and Biases

In [14]:
def relu(z): #relu kendisine girilen değerler negatifse sıfır yapar, pozitifse olduğu gibi döndürür, gizli katmanlarda kullanılacaktır
    return np.maximum(0, z)

def sigmoid(z):#çıktı katmanında kullanılır
    return 1 / (1 + np.exp(-z))

weights_hidden = np.array([[0.1, 0.2], [0.3, 0.4], [0.5, 0.6], [0.7, 0.8], [0.1, 0.2],
                           [0.3, 0.4], [0.5, 0.6], [0.7, 0.8]]) #ağırlıkları rastgele oluşturduk

biases_hidden = np.array([0.1, 0.2])

weights_output = np.array([[0.3], [0.4]]) #ikinci katmandan outputa gidecek ağırlıklar da olacak

bias_output = np.array([0.5]) #gizli katmandan outputa giden bias değerleri de olacak

# Forward Propagation

In [15]:
def forward_pass(X, weights_hidden, biases_hidden, weights_output, bias_output):

    z_hidden = np.dot(X, weights_hidden) + biases_hidden #girdilerimiz var, ağırlıklarımızla matris çarpımı yapıyoruz, üstüne bias değerlerini ekliyoruz
    a_hidden = relu(z_hidden)#birinci katman yani gizli katmandaki net girdileri ve aktivasyonları hesaplar
    print(a_hidden[:5].flatten()) #relu aktivasyon fonksiyonundan geçen değerler
    z_output = np.dot(a_hidden, weights_output) + bias_output#gizli katmandan gelen aktivasyon değerini alır, outputa giden ağırlıkları alır, outputa giden bias değerlerini alır
    a_output = sigmoid(z_output) #aldığı parametreleri sigmoid fonksiyonundan geçirip inputunu üretir

    return a_output

In [16]:
y_pred_train = forward_pass(X_train, weights_hidden, biases_hidden, weights_output, bias_output)#ileri yayılım işlemi için fonksiyon, aldığı parametreler sonucu tahmin edilen değerleri üretir

[0.         0.         2.66280187 3.24627948 0.         0.
 0.0758077  0.         2.49128411 3.32611873]


In [17]:
print(y_pred_train[:5].flatten()) #tahmin edilen değerlerin ilk beş tanesine göz atalım, flatten kullanılmasının sebebi  çıktıyı vektör formunda göstermek içindir

[0.62245933 0.93069099 0.62245933 0.62778879 0.92942123]


# Train Binary Log Loss

In [18]:
def binary_log_loss(y_true, y_pred): #elimizde tahmin edilen değerler var, gerçek değerlerle bunları kıyaslayarak binary log loss ile bunu hesaplayabiliriz
    epsilon = 1e-15 #manuel şekilde hesaplama yaptığımız için ortaya çıkabilecek logaritma fonksiyonundan kaynaklı matematiksel hataların önüne geçmek içindir
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

In [19]:
loss_train = binary_log_loss(y_train, y_pred_train) #modeli train seti üzerinden eğitiyoruz ve validasyon seti üzerinde test ediyoruz, elde ettiğimiz hata train setinin hatası

print("Train Binary log loss:", loss_train)

Train Binary log loss: 1.035331967841528


# Validation Binary Log Loss

In [20]:
y_pred_val = forward_pass(X_val, weights_hidden, biases_hidden, weights_output, bias_output) #train setinden öğrenmiş varsaydığımız değerleri ve bağımsız değişkenleri parametre olarak alır

[0.89939167 1.20405664 0.         0.         0.         0.
 0.         0.         0.4276528  0.61548067]


In [21]:
loss_val = binary_log_loss(y_val, y_pred_val) #validasyon setine ilişkin tahmin edilen değerler ve gerçek değerleri parametre olarak alır

print("Validation Set Binary log loss:", loss_val) #validasyon setinin loss değeri
#pratikte train ve validasyon seti karşılaştırıldığında genelde validasyon hatasının(loss unun) daha yüksek çıkmasını bekleriz

Validation Set Binary log loss: 1.092272605688249
